In [ ]:
puts `ls -l`

In [15]:
SOURCE='./raw_data/bv-kg-20260617.large'.freeze

(irb): warning: already initialized constant Object::SOURCE


"./raw_data/bv-kg-20260617.large"

In [ ]:
puts `head -2 #{SOURCE}`

In [ ]:
puts `head -5 ./maps/2026-biovista-drugs.map`

In [16]:
require 'net/http'
require 'uri'
require 'json'

$hpo_cache = {}

def get_hpo_label(hp_code)
  return $hpo_cache[hp_code] if $hpo_cache.key?(hp_code)
  local   = hp_code.tr(':', '_').then { |s| s.start_with?('HP_') ? s : "HP_#{s}" }
  iri     = "http://purl.obolibrary.org/obo/#{local}"
  encoded = URI.encode_uri_component(URI.encode_uri_component(iri))
  uri     = URI("https://www.ebi.ac.uk/ols4/api/ontologies/hp/terms/#{encoded}")
  response = Net::HTTP.get_response(uri)
  result = if response.is_a?(Net::HTTPSuccess)
    data = JSON.parse(response.body)
    data['label'] || "no HPO match found for #{local}"
  else
    "no HPO match found for #{local}"
  end
  $hpo_cache[hp_code] = result
rescue => e
  $hpo_cache[hp_code] = "no HPO match found for #{local}"
end

:get_hpo_label

In [17]:
require 'linkeddata'
require 'csv'

graphing_errors = File.open('./graph/2026_phenotype-drug-errors.txt', 'w')

SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

# Pre-index drug mappings by drug_id for O(1) lookup instead of O(n) linear scan
drug_mappings_index = {}
CSV.foreach('./maps/2026-biovista-drugs.map', headers: true) do |row|
  # extract the ID from the mesh URI so lookup is direct
  if row['biovista_meshid'] =~ /\/([^\/]+)$/
    drug_mappings_index[$1] = row
  end
end

failures = {}

# Open the output file ONCE, outside the loop
File.open('./graph/2026_drug-phenotype.nq.large', 'w') do |f|
  writer = RDF::Writer.for(:nquads).new(f)
  
  CSV.foreach(SOURCE, col_sep: "\t", quote_char: '"',
    liberal_parsing: true, headers: true) do |row|

    next unless (["Drug", "Compound"].include?(row['type_1']) && row['type_2'] == "Human Phenotype") ||
                (row['type_1'] == "Human Phenotype" && ["Drug", "Compound"].include?(row['type_2']))

    if row['type_1'] == "Human Phenotype"
      pheno_id = row['id_1']
      pheno_label = row['name_1']
      drug_id = row['id_2']
    else
      pheno_id = row['id_2']
      pheno_label = row['name_2']
      drug_id = row['id_1']
    end

    score    = row['score']
    evidence = row['url']

    if pheno_id.match(%r{(\d+)})
      hpo_num  = pheno_id.match(%r{(\d+)})[1]
      hpo      = RDF::URI.new("http://purl.obolibrary.org/obo/HP_#{hpo_num}")
      pheno_id = "HP_#{hpo_num}"
    else
      warn "NO MATCH FOR PHENO ID #{pheno_id}"
      graphing_errors.write("Pheno lookup failed #{pheno_id}\n")
      next
    end

    # O(1) hash lookup instead of O(n) linear scan with regex
    drug = drug_mappings_index[drug_id]

    unless drug
      unless failures[drug_id]
        failures[drug_id] = true
        warn "drug lookup failed #{drug_id}"
        graphing_errors.write("drug lookup failed #{drug_id}\n")
      end
      next
    end

    drug_cid = drug['CID']
    if drug_cid =~ /SUBSTANCE_(\d+)/
      pubchem_uri = RDF::URI.new("https://pubchem.ncbi.nlm.nih.gov/substance/#{$1}")
    else
      pubchem_uri = RDF::URI.new("https://pubchem.ncbi.nlm.nih.gov/compound/#{drug_cid}")
    end
    pubchem_type      = RDF::URI.new("http://semanticscience.org/resource/CHEMINF_000302")
    pubchem_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Drug")
    iupac_drug_label  = RDF::Literal.new(drug['IUPACname'])
    original_drug     = RDF::Literal.new(drug['biovista_meshid'])

    hpo_type      = RDF::URI.new("http://edamontology.org/data_3275")
    hpo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Phenotype")
    hpo_label = RDF::Literal.new(get_hpo_label(pheno_id))

    context_uri     = RDF::URI.new("urn:simpathic:context:bv_#{pheno_id}_#{drug_id}")
    general_context = RDF::URI.new("urn:simpathic:context:all_metadata")

    # Build a fresh repo per iteration (you were already doing this correctly)
    graph = RDF::Repository.new

    graph << RDF::Statement.new(pubchem_uri, SIMPATHIC['associated-with'], hpo,         graph_name: context_uri)
    graph << RDF::Statement.new(hpo,         SIMPATHIC['associated-with'], pubchem_uri, graph_name: context_uri)

    graph << RDF::Statement.new(pubchem_uri,       RDFS.label,               iupac_drug_label,                    graph_name: context_uri)
    graph << RDF::Statement.new(pubchem_uri,       RDF.type,                 pubchem_type,                        graph_name: context_uri)
    graph << RDF::Statement.new(pubchem_uri,       RDF.type,                 pubchem_core_type,                   graph_name: context_uri)
    graph << RDF::Statement.new(pubchem_type,      RDFS.label,               RDF::Literal.new("PubChem"),         graph_name: context_uri)
    graph << RDF::Statement.new(pubchem_core_type, RDFS.label,               RDF::Literal.new("Drug"),            graph_name: context_uri)
    graph << RDF::Statement.new(pubchem_uri,       SIMPATHIC['original-id'], original_drug,                       graph_name: context_uri)

    #  THIS NEEDS TO BE DELETED FROM THE DB
    # graph << RDF::Statement.new(hpo,          RDFS.label,               RDF::Literal.new("HPO Phenotype Identifier"), graph_name: context_uri)

    graph << RDF::Statement.new(hpo,          RDFS.label,               hpo_label,                           graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          RDF.type,                 hpo_type,                                     graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          RDF.type,                 hpo_core_type,                                graph_name: context_uri)
    graph << RDF::Statement.new(hpo_type,     RDFS.label,               RDF::Literal.new("HPO Ontology Term"),        graph_name: context_uri)
    graph << RDF::Statement.new(hpo_core_type,RDFS.label,               RDF::Literal.new("Phenotype"),                graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          SIMPATHIC['original-id'], RDF::Literal.new(pheno_id),                   graph_name: context_uri)

    graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'],      RDF::Literal.new("Biovista"),             graph_name: general_context)
    graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'],         RDF::URI.new(evidence),                   graph_name: general_context)
    graph << RDF::Statement.new(context_uri, SIMPATHIC['score'],            RDF::Literal.new(score),                  graph_name: general_context)
    graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'],  RDF::Literal.new("ASSOCIATED_WITH"),      graph_name: general_context)

    # Write directly to the already-open writer — no file open/close overhead
    graph.each_statement { |stmt| writer << stmt }
  end

  writer.flush
end

graphing_errors.close
puts "RDF quads written"

(irb):5: warning: already initialized constant Object::SIMPATHIC
(irb):5: warning: previous definition of SIMPATHIC was here
(irb):6: warning: already initialized constant Object::RDFS
(irb):6: warning: previous definition of RDFS was here
drug lookup failed 4901d7c1b4e2aef6913d880bfc91240e
drug lookup failed C032523
drug lookup failed C036309
drug lookup failed C050199
drug lookup failed C065640
drug lookup failed C079420
drug lookup failed C091590
drug lookup failed C103494
drug lookup failed C104196
drug lookup failed C545824
drug lookup failed C570710
drug lookup failed D000069896
drug lookup failed D000075462
drug lookup failed D000086663
drug lookup failed D000515
drug lookup failed D000590
drug lookup failed D000990
drug lookup failed D001647
drug lookup failed D001905
drug lookup failed D002772
drug lookup failed D003176
drug lookup failed D003181
drug lookup failed D005620
drug lookup failed D006065
drug lookup failed D006820
drug lookup failed D007072
drug lookup failed D0070

RDF quads written
